# ingest_api\nGenerated from 01_ingestion/ingest_api.py for Databricks notebook import.\n

In [ ]:
"""Production-oriented generic API ingestion adapter.\n\nFeatures:\n- Config-driven endpoint resolution\n- Config-driven auth\n- Retry support with backoff and Retry-After handling\n- GET and POST support\n- Cursor and offset pagination\n- Strict validation\n- Structured logging\n- Better error context\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport logging\nimport os\nimport time\nfrom dataclasses import dataclass\nfrom typing import Any, Mapping\n\nimport requests\nfrom requests.auth import HTTPBasicAuth\n\nlogger = logging.getLogger(__name__)\n\n_RETRY_STATUS_CODES = {429, 500, 502, 503, 504}\n_DATA_ENVELOPE_KEYS = ["data", "items", "results", "records", "value", "content"]\n_SUPPORTED_HTTP_METHODS = {"GET", "POST"}\n_SUPPORTED_AUTH_TYPES = {"none", "bearer", "api_key", "basic"}\n_SUPPORTED_PAGINATION_TYPES = {"none", "cursor", "offset"}\n\n\nclass ApiIngestionConfigError(ValueError):\n    """Raised when API ingestion configuration is invalid."""\n\n\nclass ApiIngestionExecutionError(RuntimeError):\n    """Raised when API ingestion execution fails."""\n\n\ndef _require_non_empty(value: Any, field_name: str) -> str:\n    if value is None:\n        raise ApiIngestionConfigError(f"{field_name} is required")\n    text = str(value).strip()\n    if not text:\n        raise ApiIngestionConfigError(f"{field_name} is required")\n    return text\n\n\ndef _safe_json_dumps(value: Any) -> str:\n    try:\n        return json.dumps(value, ensure_ascii=False, default=str)\n    except Exception:\n        return repr(value)\n\n\ndef _get_nested_value(payload: Any, path: str) -> Any:\n    """Get nested value from dict using dot notation, e.g. 'data.items'."""\n    current = payload\n    for part in path.split("."):\n        if not isinstance(current, dict):\n            return None\n        current = current.get(part)\n    return current\n\n\n@dataclass(frozen=True)\nclass ApiProfile:\n    base_url: str | None\n    auth_type: str\n    timeout_seconds: int\n    max_retries: int\n    retry_backoff_seconds: float\n    http_method: str\n\n    token_env: str | None = None\n    api_key_env: str | None = None\n    api_key_header: str | None = None\n    user_env: str | None = None\n    password_env: str | None = None\n\n    @classmethod\n    def from_dict(cls, api_profile: Mapping[str, Any]) -> "ApiProfile":\n        if "timeout_seconds" not in api_profile:\n            raise ApiIngestionConfigError("api_profile.timeout_seconds must be configured")\n\n        auth_type = str(api_profile.get("auth_type", "none")).strip().lower()\n        if auth_type not in _SUPPORTED_AUTH_TYPES:\n            raise ApiIngestionConfigError(\n                f"Unsupported auth_type '{auth_type}'. Supported: {sorted(_SUPPORTED_AUTH_TYPES)}"\n            )\n\n        http_method = str(api_profile.get("http_method", "GET")).strip().upper()\n        if http_method not in _SUPPORTED_HTTP_METHODS:\n            raise ApiIngestionConfigError(\n                f"Unsupported http_method '{http_method}'. Supported: {sorted(_SUPPORTED_HTTP_METHODS)}"\n            )\n\n        timeout_seconds = int(api_profile["timeout_seconds"])\n        if timeout_seconds <= 0:\n            raise ApiIngestionConfigError("api_profile.timeout_seconds must be > 0")\n\n        max_retries = int(api_profile.get("max_retries", 2))\n        if max_retries < 0:\n            raise ApiIngestionConfigError("api_profile.max_retries must be >= 0")\n\n        retry_backoff_seconds = float(api_profile.get("retry_backoff_seconds", 2.0))\n        if retry_backoff_seconds < 0:\n            raise ApiIngestionConfigError("api_profile.retry_backoff_seconds must be >= 0")\n\n        base_url = api_profile.get("base_url")\n        base_url_text = str(base_url).strip() if base_url else None\n\n        return cls(\n            base_url=base_url_text,\n            auth_type=auth_type,\n            timeout_seconds=timeout_seconds,\n            max_retries=max_retries,\n            retry_backoff_seconds=retry_backoff_seconds,\n            http_method=http_method,\n            token_env=str(api_profile.get("token_env")).strip() if api_profile.get("token_env") else None,\n            api_key_env=str(api_profile.get("api_key_env")).strip() if api_profile.get("api_key_env") else None,\n            api_key_header=str(api_profile.get("api_key_header", "X-API-Key")).strip(),\n            user_env=str(api_profile.get("user_env")).strip() if api_profile.get("user_env") else None,\n            password_env=str(api_profile.get("password_env")).strip() if api_profile.get("password_env") else None,\n        )\n\n\n@dataclass(frozen=True)\nclass SourceConfig:\n    api_endpoint: str\n    source_system: str | None = None\n    source_entity: str | None = None\n    response_data_key: str | None = None\n\n    @classmethod\n    def from_dict(cls, source: Mapping[str, Any]) -> "SourceConfig":\n        response_data_key = source.get("response_data_key")\n        return cls(\n            api_endpoint=_require_non_empty(source.get("api_endpoint"), "source.api_endpoint"),\n            source_system=str(source.get("source_system")).strip() if source.get("source_system") else None,\n            source_entity=str(source.get("source_entity")).strip() if source.get("source_entity") else None,\n            response_data_key=str(response_data_key).strip() if response_data_key else None,\n        )\n\n    @property\n    def source_id(self) -> str:\n        system = self.source_system or "unknown_system"\n        entity = self.source_entity or "unknown_entity"\n        return f"{system}.{entity}"\n\n\n@dataclass(frozen=True)\nclass PaginationConfig:\n    pagination_type: str\n    next_page_key: str\n    next_page_param: str\n    page_size_param: str\n    page_size: int\n    max_pages: int\n    offset_param: str\n\n    @classmethod\n    def from_source_options(cls, source_options: Mapping[str, Any] | None) -> "PaginationConfig":\n        opts = source_options or {}\n        pagination_type = str(opts.get("pagination_type", "none")).strip().lower()\n        if pagination_type not in _SUPPORTED_PAGINATION_TYPES:\n            raise ApiIngestionConfigError(\n                f"Unsupported pagination_type '{pagination_type}'. Supported: {sorted(_SUPPORTED_PAGINATION_TYPES)}"\n            )\n\n        page_size = int(opts.get("page_size", 500))\n        if page_size <= 0:\n            raise ApiIngestionConfigError("source_options.page_size must be > 0")\n\n        max_pages = int(opts.get("max_pages", 100))\n        if max_pages <= 0:\n            raise ApiIngestionConfigError("source_options.max_pages must be > 0")\n\n        return cls(\n            pagination_type=pagination_type,\n            next_page_key=str(opts.get("next_page_key", "next_page_token")).strip(),\n            next_page_param=str(opts.get("next_page_param", "page_token")).strip(),\n            page_size_param=str(opts.get("page_size_param", "limit")).strip(),\n            page_size=page_size,\n            max_pages=max_pages,\n            offset_param=str(opts.get("offset_param", "offset")).strip(),\n        )\n\n\n@dataclass(frozen=True)\nclass RequestOptions:\n    body: dict[str, Any] | None\n    envelope_key: str | None\n\n    @classmethod\n    def from_source_and_options(\n        cls,\n        source: Mapping[str, Any],\n        source_options: Mapping[str, Any] | None,\n    ) -> "RequestOptions":\n        opts = source_options or {}\n        body = opts.get("request_body")\n        if body is not None and not isinstance(body, dict):\n            raise ApiIngestionConfigError("source_options.request_body must be a dict when provided")\n\n        envelope_key = source.get("response_data_key") or opts.get("response_data_key")\n        envelope_key_text = str(envelope_key).strip() if envelope_key else None\n\n        return cls(\n            body=body,\n            envelope_key=envelope_key_text,\n        )\n\n\ndef _resolve_api_url(api_profile: ApiProfile, source: SourceConfig) -> str:\n    endpoint = source.api_endpoint\n\n    if endpoint.startswith("http://") or endpoint.startswith("https://"):\n        return endpoint\n\n    if not api_profile.base_url:\n        raise ApiIngestionConfigError(\n            "api_profile.base_url is required when source.api_endpoint is relative"\n        )\n\n    return f"{api_profile.base_url.rstrip('/')}/{endpoint.lstrip('/')}"\n\n\ndef _resolve_auth_headers_and_auth(\n    api_profile: ApiProfile,\n    headers: Mapping[str, Any] | None = None,\n) -> tuple[dict[str, str], HTTPBasicAuth | None]:\n    final_headers = {str(k): str(v) for k, v in (headers or {}).items()}\n    auth = None\n\n    if api_profile.auth_type == "none":\n        return final_headers, None\n\n    if api_profile.auth_type == "bearer":\n        token_env = _require_non_empty(api_profile.token_env, "api_profile.token_env")\n        token = os.getenv(token_env, "").strip()\n        if not token:\n            raise ApiIngestionConfigError(\n                f"Environment variable '{token_env}' is required for bearer auth"\n            )\n        final_headers.setdefault("Authorization", f"Bearer {token}")\n        return final_headers, None\n\n    if api_profile.auth_type == "api_key":\n        key_env = _require_non_empty(api_profile.api_key_env, "api_profile.api_key_env")\n        key_header = _require_non_empty(api_profile.api_key_header, "api_profile.api_key_header")\n        api_key = os.getenv(key_env, "").strip()\n        if not api_key:\n            raise ApiIngestionConfigError(\n                f"Environment variable '{key_env}' is required for api_key auth"\n            )\n        final_headers.setdefault(key_header, api_key)\n        return final_headers, None\n\n    if api_profile.auth_type == "basic":\n        user_env = _require_non_empty(api_profile.user_env, "api_profile.user_env")\n        password_env = _require_non_empty(api_profile.password_env, "api_profile.password_env")\n        user = os.getenv(user_env, "").strip()\n        password = os.getenv(password_env, "").strip()\n        if not user or not password:\n            raise ApiIngestionConfigError(\n                f"Environment variables '{user_env}' and '{password_env}' are required for basic auth"\n            )\n        auth = HTTPBasicAuth(user, password)\n        return final_headers, auth\n\n    raise ApiIngestionConfigError(f"Unsupported auth_type '{api_profile.auth_type}'")\n\n\ndef _extract_records(payload: Any, envelope_key: str | None = None) -> list[dict[str, Any]]:\n    """Extract a flat list of dict records from the API payload."""\n    if isinstance(payload, list):\n        if all(isinstance(item, dict) for item in payload):\n            return list(payload)\n        return [{"value": item} for item in payload]\n\n    if isinstance(payload, dict):\n        if envelope_key:\n            nested = _get_nested_value(payload, envelope_key)\n            if nested is None:\n                raise ApiIngestionExecutionError(\n                    f"Configured response_data_key '{envelope_key}' not found in API response"\n                )\n            if isinstance(nested, list):\n                if all(isinstance(item, dict) for item in nested):\n                    return list(nested)\n                return [{"value": item} for item in nested]\n            if isinstance(nested, dict):\n                return [nested]\n            return [{"value": nested}]\n\n        for key in _DATA_ENVELOPE_KEYS:\n            value = payload.get(key)\n            if isinstance(value, list):\n                if all(isinstance(item, dict) for item in value):\n                    return list(value)\n                return [{"value": item} for item in value]\n\n        return [payload]\n\n    return []\n\n\ndef _parse_response_json(response: requests.Response) -> Any:\n    content_type = response.headers.get("Content-Type", "")\n    if "json" not in content_type.lower() and "javascript" not in content_type.lower():\n        snippet = response.text[:500]\n        raise ApiIngestionExecutionError(\n            f"Expected JSON response but got Content-Type={content_type!r}. "\n            f"Response snippet={snippet!r}"\n        )\n\n    try:\n        return response.json()\n    except ValueError as exc:\n        snippet = response.text[:500]\n        raise ApiIngestionExecutionError(\n            f"Response advertised JSON but could not be parsed. Snippet={snippet!r}"\n        ) from exc\n\n\ndef _sleep_before_retry(response: requests.Response | None, base_backoff_seconds: float, attempt: int) -> None:\n    if response is not None:\n        retry_after = response.headers.get("Retry-After")\n        if retry_after:\n            try:\n                time.sleep(float(retry_after))\n                return\n            except ValueError:\n                pass\n\n    time.sleep(base_backoff_seconds * (attempt + 1))\n\n\ndef _send_request(\n    session: requests.Session,\n    method: str,\n    url: str,\n    headers: Mapping[str, str],\n    params: Mapping[str, Any],\n    json_body: Mapping[str, Any] | None,\n    timeout: int,\n    auth: HTTPBasicAuth | None,\n) -> requests.Response:\n    if method == "GET":\n        return session.get(url, headers=headers, params=params, timeout=timeout, auth=auth)\n    if method == "POST":\n        return session.post(url, headers=headers, params=params, json=json_body, timeout=timeout, auth=auth)\n\n    raise ApiIngestionConfigError(f"Unsupported HTTP method '{method}'")\n\n\ndef ingest_api(\n    source: Mapping[str, Any],\n    api_profile: Mapping[str, Any],\n    headers: Mapping[str, Any] | None = None,\n    params: Mapping[str, Any] | None = None,\n    source_options: Mapping[str, Any] | None = None,\n) -> list[dict[str, Any]]:\n    """Ingest records from a REST API endpoint with retry and pagination support."""\n    profile = ApiProfile.from_dict(api_profile)\n    source_cfg = SourceConfig.from_dict(source)\n    pagination = PaginationConfig.from_source_options(source_options)\n    request_options = RequestOptions.from_source_and_options(source, source_options)\n\n    endpoint = _resolve_api_url(profile, source_cfg)\n    final_headers, auth = _resolve_auth_headers_and_auth(profile, headers=headers)\n\n    current_params: dict[str, Any] = dict(params or {})\n    if pagination.pagination_type in {"cursor", "offset"} and pagination.page_size_param:\n        current_params[pagination.page_size_param] = pagination.page_size\n\n    all_records: list[dict[str, Any]] = []\n    offset = 0\n    seen_tokens: set[str] = set()\n    started_at = time.time()\n\n    logger.info(\n        "Starting API ingestion source_id=%s method=%s endpoint=%s pagination_type=%s",\n        source_cfg.source_id,\n        profile.http_method,\n        endpoint,\n        pagination.pagination_type,\n    )\n\n    with requests.Session() as session:\n        for page_num in range(1, pagination.max_pages + 1):\n            if pagination.pagination_type == "offset":\n                current_params[pagination.offset_param] = offset\n\n            response: requests.Response | None = None\n\n            for attempt in range(profile.max_retries + 1):\n                try:\n                    logger.debug(\n                        "API request source_id=%s page_num=%s attempt=%s params=%s",\n                        source_cfg.source_id,\n                        page_num,\n                        attempt + 1,\n                        _safe_json_dumps(current_params),\n                    )\n\n                    response = _send_request(\n                        session=session,\n                        method=profile.http_method,\n                        url=endpoint,\n                        headers=final_headers,\n                        params=current_params,\n                        json_body=request_options.body,\n                        timeout=profile.timeout_seconds,\n                        auth=auth,\n                    )\n\n                    if response.status_code in _RETRY_STATUS_CODES and attempt < profile.max_retries:\n                        logger.warning(\n                            "Retryable API response source_id=%s page_num=%s status_code=%s attempt=%s",\n                            source_cfg.source_id,\n                            page_num,\n                            response.status_code,\n                            attempt + 1,\n                        )\n                        _sleep_before_retry(response, profile.retry_backoff_seconds, attempt)\n                        continue\n\n                    response.raise_for_status()\n                    break\n\n                except requests.RequestException as exc:\n                    if attempt >= profile.max_retries:\n                        raise ApiIngestionExecutionError(\n                            f"API request failed source_id={source_cfg.source_id} "\n                            f"endpoint={endpoint} page_num={page_num} "\n                            f"after {profile.max_retries + 1} attempt(s): {exc}"\n                        ) from exc\n\n                    logger.warning(\n                        "Transient API request failure source_id=%s page_num=%s attempt=%s error=%s",\n                        source_cfg.source_id,\n                        page_num,\n                        attempt + 1,\n                        str(exc),\n                    )\n                    _sleep_before_retry(response, profile.retry_backoff_seconds, attempt)\n\n            if response is None:\n                raise ApiIngestionExecutionError(\n                    f"API ingestion failed with unknown error for source_id={source_cfg.source_id}"\n                )\n\n            payload = _parse_response_json(response)\n            records = _extract_records(payload, envelope_key=request_options.envelope_key)\n            all_records.extend(records)\n\n            logger.info(\n                "Fetched API page source_id=%s page_num=%s records=%s total_records=%s",\n                source_cfg.source_id,\n                page_num,\n                len(records),\n                len(all_records),\n            )\n\n            if pagination.pagination_type == "cursor":\n                next_token = None\n                if isinstance(payload, dict):\n                    next_token = _get_nested_value(payload, pagination.next_page_key)\n\n                if not next_token:\n                    break\n\n                next_token_str = str(next_token)\n                if next_token_str in seen_tokens:\n                    raise ApiIngestionExecutionError(\n                        f"Detected repeated cursor token for source_id={source_cfg.source_id}. "\n                        f"Token={next_token_str!r}"\n                    )\n\n                seen_tokens.add(next_token_str)\n                current_params[pagination.next_page_param] = next_token\n\n            elif pagination.pagination_type == "offset":\n                if not records:\n                    break\n                offset += len(records)\n\n            else:\n                break\n\n    elapsed = round(time.time() - started_at, 3)\n    logger.info(\n        "Completed API ingestion source_id=%s endpoint=%s total_records=%s elapsed_seconds=%s",\n        source_cfg.source_id,\n        endpoint,\n        len(all_records),\n        elapsed,\n    )\n\n    return all_records\n